<a href="https://colab.research.google.com/github/Servejit/AI-video/blob/main/wan_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Wan 2.1 Free Colab Video Engine

This notebook runs open-source **Wan 2.1 T2V 1.3B** in your Google Colab runtime and exposes a temporary HTTPS endpoint for the Streamlit app.

**Setup note:** the first cell removes a conflicting optional `torchao` package and pins compatible Diffusers/Transformers versions. When it finishes, use **Runtime → Restart session**, then rerun the notebook cells in order.

- Choose **Runtime → Change runtime type → T4 GPU** if available.
- Free GPU availability, session duration, and resource limits are controlled by Google and are not guaranteed.
- Keep this notebook running while your Streamlit app generates clips.
- Keep the endpoint URL and access token private.
- First-time setup downloads several gigabytes of model files.
- No paid API or Hugging Face token is required for this public model.


In [ ]:
# Repair the Colab dependency mismatch, then install compatible Wan 2.1 packages.
# IMPORTANT: after this cell completes, choose Runtime > Restart session, then run the GPU-check cell again.
# Keep Colab's preinstalled torch/CUDA build; remove the incompatible optional torchao package.
!pip -q uninstall -y torchao
!pip -q install --no-cache-dir "diffusers==0.32.2" "transformers==4.49.0" "accelerate==1.3.0" ftfy flask imageio imageio-ffmpeg
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("No GPU is attached. In Colab select Runtime > Change runtime type > T4 GPU (if available), then rerun this cell.")
print("GPU:", torch.cuda.get_device_name(0))
try:
    import torchao
    raise RuntimeError("torchao is still installed. Restart the Colab runtime and rerun this cell.")
except ImportError:
    print("torchao conflict check: clear")


In [ ]:
import os
import time
import uuid
import secrets
import threading
import subprocess
from pathlib import Path

import torch
from flask import Flask, jsonify, request, send_file
from diffusers import AutoencoderKLWan, WanPipeline
from diffusers.schedulers.scheduling_unipc_multistep import UniPCMultistepScheduler
from diffusers.utils import export_to_video

MODEL_ID = "Wan-AI/Wan2.1-T2V-1.3B-Diffusers"
PORT = 7860
ACCESS_TOKEN = secrets.token_urlsafe(32)
JOB_ROOT = Path("/content/wan21_jobs")
JOB_ROOT.mkdir(parents=True, exist_ok=True)
jobs = {}
jobs_lock = threading.Lock()
generation_lock = threading.Lock()

print("Loading Wan 2.1 1.3B. First run downloads model files; please wait...")
# FP32 VAE + FP16 weights is a practical setting for Colab T4 GPUs.
vae = AutoencoderKLWan.from_pretrained(MODEL_ID, subfolder="vae", torch_dtype=torch.float32)
pipe = WanPipeline.from_pretrained(MODEL_ID, vae=vae, torch_dtype=torch.float16)
pipe.scheduler = UniPCMultistepScheduler(
    prediction_type="flow_prediction",
    use_flow_sigmas=True,
    num_train_timesteps=1000,
    flow_shift=3.0,
)
pipe.enable_model_cpu_offload()
print("Wan 2.1 loaded.")

app = Flask("wan21_colab")

def authorized():
    return secrets.compare_digest(
        str(request.headers.get("X-API-Token", "")),
        ACCESS_TOKEN,
    )

@app.get("/health")
def health():
    if not authorized():
        return jsonify({"error": "Unauthorized"}), 401
    return jsonify({"ok": True, "model": MODEL_ID})

@app.post("/generate")
def generate():
    if not authorized():
        return jsonify({"error": "Unauthorized"}), 401
    data = request.get_json(silent=True) or {}
    prompt = str(data.get("prompt", "")).strip()
    if not prompt:
        return jsonify({"error": "Prompt is required"}), 400
    if len(prompt) > 2000:
        return jsonify({"error": "Prompt is too long (maximum 2000 characters)"}), 400

    job_id = uuid.uuid4().hex
    with jobs_lock:
        jobs[job_id] = {"status": "queued", "error": "", "path": ""}
    def worker():
        try:
            with generation_lock:
                with jobs_lock:
                    jobs[job_id]["status"] = "running"
                out_path = JOB_ROOT / f"{job_id}.mp4"
                # 33 frames = 4*k+1, 480p landscape; Streamlit crops/resizes to the chosen format.
                with torch.inference_mode():
                    result = pipe(
                        prompt=prompt,
                        negative_prompt="subtitles, captions, text, letters, logos, watermarks, blurry, distorted anatomy, static frame",
                        width=832,
                        height=480,
                        num_frames=33,
                        num_inference_steps=20,
                        guidance_scale=5.0,
                    )
                export_to_video(result.frames[0], str(out_path), fps=8)
                if not out_path.exists() or out_path.stat().st_size < 1024:
                    raise RuntimeError("Video export failed or produced an empty file.")
                with jobs_lock:
                    jobs[job_id]["status"] = "completed"
                    jobs[job_id]["path"] = str(out_path)
        except Exception as exc:
            with jobs_lock:
                jobs[job_id]["status"] = "failed"
                jobs[job_id]["error"] = str(exc)[:1500]
            print(f"Job {job_id} failed:", repr(exc))
        finally:
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
    threading.Thread(target=worker, daemon=True).start()
    return jsonify({"job_id": job_id, "status": "queued"}), 202

@app.get("/status/<job_id>")
def status(job_id):
    if not authorized():
        return jsonify({"error": "Unauthorized"}), 401
    with jobs_lock:
        item = jobs.get(job_id)
        if not item:
            return jsonify({"error": "Unknown job ID"}), 404
        return jsonify({"status": item["status"], "error": item["error"]})

@app.get("/download/<job_id>")
def download(job_id):
    if not authorized():
        return jsonify({"error": "Unauthorized"}), 401
    with jobs_lock:
        item = jobs.get(job_id)
        if not item or item["status"] != "completed":
            return jsonify({"error": "Video is not ready"}), 404
        path = item["path"]
    return send_file(path, mimetype="video/mp4", as_attachment=True, download_name=f"wan21_{job_id}.mp4")

def run_server():
    app.run(host="0.0.0.0", port=PORT, debug=False, use_reloader=False, threaded=True)

threading.Thread(target=run_server, daemon=True).start()
time.sleep(2)

print("\n================ COPY THESE INTO YOUR STREAMLIT APP ================")
print("ACCESS TOKEN (keep private):")
print(ACCESS_TOKEN)
print("\nThe HTTPS endpoint URL appears below after Cloudflare starts.")
print("Leave this cell running. Copy the https://....trycloudflare.com URL from its output.")
print("====================================================================\n")
# This command stays active to keep the temporary tunnel open.
# The URL changes when the Colab runtime restarts.
tunnel = subprocess.run(
    ["/usr/local/bin/cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"],
    check=False,
)
